# Plot All Figures
Run **after** training is complete for each figure.
Each section is independent — run only the sections whose data is ready.

No GPU needed — CPU runtime is fine.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/stoch_depth'
print('Drive mounted.')

---
## Figure 3 — CIFAR-10 Test Error & Training Loss

In [ ]:
import json

def load_log(path):
    with open(path) as f:
        log = json.load(f)
    return {
        'epochs':     [r['epoch']      for r in log],
        'test_err':   [r['test_err']   for r in log],
        'train_loss': [r['train_loss'] for r in log],
    }

stoch = load_log(f'{ROOT}/fig3/stochastic/logs/cifar10_pL0.5_modelinear_n18.json')
base  = load_log(f'{ROOT}/fig3/baseline/logs/cifar10_pL1.0_modeconstant_n18.json')
print(f'Stochastic epochs: {len(stoch["epochs"])}')
print(f'Baseline   epochs: {len(base["epochs"])}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

# Test error lines (left axis) — match paper colors
l1, = ax1.plot(base['epochs'],  base['test_err'],
               color='#d62728', linewidth=1.2,
               label='Test Error with Constant Depth')
l2, = ax1.plot(stoch['epochs'], stoch['test_err'],
               color='#1f77b4', linewidth=1.2,
               label='Test Error with Stochastic Depth')

# Training loss lines (right axis) — match paper colors
l3, = ax2.semilogy(base['epochs'],  base['train_loss'],
                   color='#e377c2', linewidth=1.0,
                   label='Training Loss with Constant Depth')
l4, = ax2.semilogy(stoch['epochs'], stoch['train_loss'],
                   color='#2ca02c', linewidth=1.0,
                   label='Training Loss with Stochastic Depth')

for m in [250, 375]:
    ax1.axvline(m, color='gray', linestyle=':', linewidth=0.8)

ax1.set_ylim(4, 20)
ax1.set_xlim(0, 500)

# Annotate best test error points
best_base_err  = min(base['test_err'])
best_stoch_err = min(stoch['test_err'])
best_base_ep   = base['epochs'][base['test_err'].index(best_base_err)]
best_stoch_ep  = stoch['epochs'][stoch['test_err'].index(best_stoch_err)]

ax1.annotate(f'{best_base_err:.2f}%',
             xy=(best_base_ep, best_base_err),
             xytext=(8, 8), textcoords='offset points',
             color='#d62728', fontsize=9, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#d62728', lw=0.8))
ax1.annotate(f'{best_stoch_err:.2f}%',
             xy=(best_stoch_ep, best_stoch_err),
             xytext=(8, -14), textcoords='offset points',
             color='#1f77b4', fontsize=9, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#1f77b4', lw=0.8))

ax1.set_xlabel('epoch', fontsize=12)
ax1.set_ylabel('test error (%)', fontsize=12)
ax2.set_ylabel('training loss', fontsize=12)
ax1.set_title('110-layer ResNet on CIFAR-10', fontsize=13)
ax1.legend([l1, l2, l3, l4], [l.get_label() for l in [l1, l2, l3, l4]],
           loc='upper right', fontsize=8)

plt.tight_layout()
out = f'{ROOT}/figures/figure3.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Baseline best:   {best_base_err:.2f}%  (paper: 6.41%)')
print(f'Stochastic best: {best_stoch_err:.2f}%  (paper: 5.25%)')
print(f'Saved: {out}')

---
## Figure 7 — Gradient Magnitude

In [ ]:
import json

def load_grad_log(path):
    with open(path) as f:
        log = json.load(f)
    return {
        'epochs':   [r['epoch']    for r in log],
        'grad_mag': [r['grad_mag'] for r in log],
    }

stoch7 = load_grad_log(f'{ROOT}/fig7/stochastic/logs/cifar10_pL0.5_modelinear_n18.json')
base7  = load_grad_log(f'{ROOT}/fig7/baseline/logs/cifar10_pL1.0_modeconstant_n18.json')
print(f'Stochastic epochs: {len(stoch7["epochs"])}')
print(f'Baseline   epochs: {len(base7["epochs"])}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(base7['epochs'],  base7['grad_mag'],  color='#d62728', linewidth=1.0,
        label='Constant Depth')
ax.plot(stoch7['epochs'], stoch7['grad_mag'], color='#1f77b4', linewidth=1.0,
        label='Stochastic Depth')

for m in [250, 375]:
    ax.axvline(m, color='gray', linestyle='--', linewidth=0.8)

ax.set_xlabel('epoch', fontsize=12)
ax.set_ylabel('mean gradient magnitude', fontsize=12)
ax.set_title('First Conv Layer — Mean Gradient Magnitude', fontsize=13)
ax.legend(fontsize=10)
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.yaxis.set_major_formatter(plt.ScalarFormatter(useMathText=True))
ax.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
out = f'{ROOT}/figures/figure7.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

---
## Figure 8 Left — p_L Sensitivity

In [ ]:
import json, os
import matplotlib.pyplot as plt

P_L_VALUES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

def best_test(log_path):
    if not os.path.exists(log_path):
        return None
    with open(log_path) as f:
        log = json.load(f)
    return min(r['test_err'] for r in log)

linear_errs, uniform_errs = [], []
for p_L in P_L_VALUES:
    if p_L == 1.0:
        lin_path = f'{ROOT}/fig3/baseline/logs/cifar10_pL1.0_modeconstant_n18.json'
        uni_path = lin_path
    else:
        lin_path = f'{ROOT}/fig8_left/linear/logs/cifar10_pL{p_L}_modelinear_n18.json'
        uni_path = f'{ROOT}/fig8_left/uniform/logs/cifar10_pL{p_L}_modeuniform_n18.json'
    linear_errs.append(best_test(lin_path))
    uniform_errs.append(best_test(uni_path))

baseline_err = best_test(f'{ROOT}/fig3/baseline/logs/cifar10_pL1.0_modeconstant_n18.json')

fig, ax = plt.subplots(figsize=(8, 5))

lin_pts = [(p, e) for p, e in zip(P_L_VALUES, linear_errs)  if e is not None]
uni_pts = [(p, e) for p, e in zip(P_L_VALUES, uniform_errs) if e is not None]

if lin_pts:
    ax.plot([x[0] for x in lin_pts], [x[1] for x in lin_pts],
            color='#1f77b4', linewidth=1.5, marker='o', markersize=5,
            label='Stochastic Depth (linear decay)')
if uni_pts:
    ax.plot([x[0] for x in uni_pts], [x[1] for x in uni_pts],
            color='#1f77b4', linewidth=1.5, linestyle='--', marker='o', markersize=5,
            label='Stochastic Depth (uniform)')
if baseline_err:
    ax.axhline(baseline_err, color='#d62728', linewidth=1.5, linestyle='--',
               label='Constant Depth')

ax.set_xlabel('survival probability p_L', fontsize=12)
ax.set_ylabel('test error (%)', fontsize=12)
ax.set_title('110-layer ResNet on CIFAR-10 with Varying Survival Probabilities', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, linestyle=':', alpha=0.5)
ax.set_xlim(0.05, 1.05)

plt.tight_layout()
out = f'{ROOT}/figures/figure8_left.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

---
## Figure 8 Right — Depth × p_L Heatmap

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt

BLOCKS   = [3, 6, 9, 12, 15, 18]
DEPTHS   = [6*n+2 for n in BLOCKS]
P_L_VALS = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

def best_test(log_path):
    if not os.path.exists(log_path):
        return float('nan')
    with open(log_path) as f:
        log = json.load(f)
    return min(r['test_err'] for r in log)

# Build matrix rows=p_L (high->low), cols=depth (low->high)
grid = np.full((len(P_L_VALS), len(BLOCKS)), float('nan'))
for j, n in enumerate(BLOCKS):
    for i, p_L in enumerate(P_L_VALS):
        mode = 'constant' if p_L == 1.0 else 'linear'
        # For 110-layer p_L=1.0 reuse Fig 3 baseline
        if n == 18 and p_L == 1.0:
            path = f'{ROOT}/fig3/baseline/logs/cifar10_pL1.0_modeconstant_n18.json'
        else:
            path = f'{ROOT}/fig8_right/logs/cifar10_pL{p_L}_mode{mode}_n{n}.json'
        grid[i, j] = best_test(path)

grid_plot    = grid[::-1]   # flip so p_L=1.0 at top
p_L_labels   = [str(p) for p in P_L_VALS[::-1]]

fig, ax = plt.subplots(figsize=(8, 5))
valid = grid_plot[~np.isnan(grid_plot)]
im = ax.imshow(grid_plot, cmap='RdYlBu_r', aspect='auto',
               vmin=valid.min() if len(valid) else 5,
               vmax=valid.max() if len(valid) else 9)

for i in range(len(P_L_VALS)):
    for j in range(len(BLOCKS)):
        val = grid_plot[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
        else:
            ax.text(j, i, '—', ha='center', va='center',
                    fontsize=9, color='gray')

ax.set_xticks(range(len(DEPTHS)))
ax.set_xticklabels(DEPTHS)
ax.set_yticks(range(len(P_L_VALS)))
ax.set_yticklabels(p_L_labels)
ax.set_xlabel('network depth (in layers)', fontsize=12)
ax.set_ylabel('p_L', fontsize=12)
ax.set_title('Test Error (%) on CIFAR-10', fontsize=13)
plt.colorbar(im, ax=ax, label='test error (%)')

plt.tight_layout()
out = f'{ROOT}/figures/figure8_right.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')